[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/induced_dipoles/induced_dipoles.ipynb)

# Induced dipoles in a cluster of atoms

An atom in an electric field polarizes: its electrons shift against its nucleus, and it carries a dipole, its polarizability times the field. In a cluster, each atom also feels the fields of the other atoms' dipoles. In a chain along the applied field, a dipole's field at its neighbours points along the applied field and adds to it, so the chain polarizes more than its atoms would apart; in a chain across the field, the neighbours' fields oppose it, and the chain polarizes less. Every dipole depends on every other, so they are found together: one solve of a field map that couples each atom to all the others.

This is the dipole interaction model Silberstein used in 1917 for the polarizability of diatomic molecules, and, with light in place of a static field, the discrete dipole approximation of Purcell and Pennypacker for the scattering of light by dust grains.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_figure
from examples.electromagnetism.induced_dipoles import render

np.set_printoptions(precision=4, suppress=True)

ga = VGA3D
mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
# Units in which four pi times the permittivity of the vacuum is one.
SPACING = 1.0
# The polarizability of each atom, a tenth of the spacing cubed: well below where the cluster would
# polarize itself without an applied field.
ALPHA = 0.1 * SPACING**3
FIELD = 1.0
ATOMS = 7

## 1. Three clusters

Seven atoms in the plane $z = 0$, one spacing apart: in a line along the applied field, which points along $x$; the same line a quarter turn across it; and a ring, its atoms turned around the centre. The three clusters are stacked as cases, so every cell after this one runs on all three at once.

In [ ]:
offsets = SPACING * (np.arange(ATOMS) - (ATOMS - 1) / 2)                        # [atoms]
along = mv.x * offsets                                                          # [atoms] Vector
across = (mv.xy * (-np.pi / 4)).exp() >> along                                  # [atoms] Vector
# A ring of atoms one spacing from their neighbours, each turned from x by its share of a full turn.
ring_radius = SPACING / (2 * np.sin(np.pi / ATOMS))
turns = 2 * np.pi * np.arange(ATOMS) / ATOMS                                    # [atoms]
ring = (mv.xy * (-turns / 2)).exp() >> (ring_radius * mv.x)                     # [atoms] Vector
cases = ["chain along the field", "chain across the field", "ring"]
positions = stack([along, across, ring])                                        # [cases, atoms] Vector

## 2. The coupling between atoms

A dipole makes a field around it, twice as strong along its own line as across it, and falling off with the cube of the distance: three times the separation times its inner product with the dipole, less the dipole times the squared distance, all over the fifth power of the distance, $\left(3\, r\, (r \cdot p) - r^2 p\right) / |r|^5$. Taken as a map from the dipole of one atom to the field it makes at another, this is a `Vector <- Vector` for every pair of atoms, the coupling.

Each atom's dipole is its polarizability times the field at it: the applied field plus the coupled fields of all the other atoms. Turned around, the applied field at an atom is its dipole over the polarizability, less the fields of the others. That is one map from the dipoles of all the atoms to the applied field at all the atoms, the response: `Vector / ALPHA` where an atom meets itself, minus the coupling between two different atoms. Built as a batch of maps over the pairs of atoms, output atom first and input atom second, `.field(0, 1)` reads the pairs as the sites of its two slots: a field map `Vector[atoms] <- Vector[atoms]`, one per case. Bound to the dipoles of the atoms, it sums over them.

In [ ]:
# One where an atom meets itself, the diagonal of the pairs; zero between two different atoms.
same = mv.scalar(np.eye(ATOMS)[..., None])                                      # [atoms, atoms] Scalar
# From each input atom to each output atom.
separation = positions[:, :, None] - positions[:, None, :]                      # [cases, atoms, atoms] Vector
squared_distance = separation.squared()                                         # [cases, atoms, atoms] Scalar
# Where an atom meets itself the separation vanishes, and with it the numerator of the dipole field; a one
# added to the distance there keeps the division finite.
distance = (squared_distance + same).square_root()                              # [cases, atoms, atoms] Scalar
# The coupling between every pair, read as the field map from the dipoles of the atoms to the fields they make.
coupling = ((3 * separation * (separation | Vector) - squared_distance * Vector) / distance**5).field(0, 1)   # [cases] Vector[atoms] <- Vector[atoms]
# One over the polarizability at every atom, on the diagonal, as a map without sites acts on a field.
response = Vector / ALPHA - coupling                                            # [cases] Vector[atoms] <- Vector[atoms]

print(response)

## 3. All dipoles at once

The applied field is the same at every atom. Solving the response against it gives the dipoles of all the atoms together: one coupled system over every pair of an atom and a direction, which no set of separate solves, one per atom, could give, since each atom's dipole depends on all the others. The mean dipole per atom, against the dipole of an isolated atom in the same field, tells how much more or less the cluster polarizes than its atoms apart.

In block-matrix notation this reads as the linear system $\left(\alpha^{-1} \mathbb{1} - \mathsf{T}\right) \mathbf{p} = \mathbf{E}_0$ of $3N$ equations, with the dipole tensors $\mathsf{T}_{ij} = \left(3\, \hat r_{ij} \hat r_{ij}^{\mathsf{T}} - \mathbb{1}\right) / r_{ij}^3$ in the off-diagonal blocks of $\mathsf{T}$ and zero blocks on its diagonal.

In [ ]:
applied = FIELD * mv.x                                                          # [] Vector, the same at every atom
dipoles = response.solve(applied)                                               # [cases] Vector[atoms]
isolated = ALPHA * FIELD * mv.x                                                 # [] Vector
mean_dipole = dipoles.batch().mean(axis=-1)                                     # [cases] Vector
gain = mean_dipole | isolated.inverse()                                         # [cases] Scalar

print("mean dipole per atom, relative to an isolated atom:")
for name, value in zip(cases, gain.to_array()):
    print(f"  {name}: {value:.4f}")

## 4. The dipoles

Each atom with its dipole, all to one scale, over the dipole an isolated atom would carry, in light grey. Along the field the dipoles grow towards the middle of the chain, where neighbours on both sides add to the field; across it they shrink most in the middle. Around the ring, the atoms where the ring runs along the field polarize more than those where it runs across it, and every dipole tilts towards the ring's tangent.

In [ ]:
save_figure(render.clusters(positions, dipoles, isolated), "induced_dipoles")

In [ ]:
# checks
# Each atom's dipole is its polarizability times the field at it: the applied field plus the coupled
# fields of all the other atoms' dipoles.
local = applied + coupling(dipoles)                                             # [cases] Vector[atoms]
np.testing.assert_allclose((dipoles - ALPHA * local).kernel, 0.0, atol=1e-13)
# The response is its own adjoint: each coupling is symmetric, and the field of one atom's dipole at
# another is the field of the other's at the one.
np.testing.assert_allclose((response.adjoint() - response).kernel, 0.0, atol=1e-13)
# A chain polarizes more than its atoms apart along the field and less across it; a ring in the plane of
# the field lies in between.
along_gain, across_gain, ring_gain = gain.to_array()
assert along_gain > ring_gain > 1 > across_gain
# Far from the polarization catastrophe, where the dipoles would hold each other up without an applied
# field and the response would have an eigenvalue of zero: every eigenvalue of the response keeps more than
# half of the one over the polarizability an isolated atom has.
assert (ALPHA * response.eigvalsh()).to_array().min() > 1 / 2